## Pydantic Basics: Creating and Using Models
Pydantic models are the foundation of data validation in Python. They use Python type annotations to define the structure and validate data at runtime.

In [1]:
from pydantic import BaseModel

In [2]:
from dataclasses import dataclass

@dataclass
class Person1():
    name:str
    age:int
    address:str

person1 = Person1(name="vishal",age=32,address="Gurgaon")
print(person1)

Person1(name='vishal', age=32, address='Gurgaon')


In [3]:
person2 = Person1(name="vishal",age=32,address=20)
print(person2)

# Data Validation is Not Happening Here.

Person1(name='vishal', age=32, address=20)


## 1. Base Model (With Data Validation)

In [4]:
class Person3(BaseModel):
    name:str
    age:int
    address:str

person3 = Person3(name="vishal",age=32,address="Gurgaon")
print(person3)

name='vishal' age=32 address='Gurgaon'


In [5]:

person4 = Person3(name="vishal",age=32,address=35)
print(person4)

ValidationError: 1 validation error for Person3
address
  Input should be a valid string [type=string_type, input_value=35, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

## 2. Model with Optional Fields
### Add optional fields using Python's Optional type:

In [6]:
from typing import Optional

class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None  # Optional with default value = None
    is_active: Optional[bool] = True  # Optional with default value = True


emp1 = Employee(id=1, name="Vishal", department="CSE")

print(emp1)

id=1 name='Vishal' department='CSE' salary=None is_active=True


In [7]:
emp2 = Employee(id=2, name="Goyal", department="IT", salary=100000000, is_active= False)
print(emp2)

# Automatic TypeCasting of Float Value.

id=2 name='Goyal' department='IT' salary=100000000.0 is_active=False


## 3. Model with List Values

In [8]:
from typing import List

class Classroom(BaseModel):
    room_number: str
    students: List[str]  # List of strings
    capacity: int


# This Includes Automatic TypeCating

In [9]:
classroom = Classroom(
    room_number= "A101",
    students= {"Vishal", "Goyal", "Data Sci."}, # () and [] will also work.
    capacity= 30.00        # This will turn into INTEGER
)
print(classroom)

room_number='A101' students=['Goyal', 'Data Sci.', 'Vishal'] capacity=30


In [10]:
try:
    invalid_val= Classroom(room_number="A1", students=["Krish",123], capacity=30)

except ValueError as e:
    print(e)

1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


## 4. Model with Nested Models
Create complex structures with nested models:



In [11]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: int

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address  # Nested model

# Create a customer with nested address
customer = Customer(
    customer_id= 100,
    name= "Vishal",
    address= {"street": "NO FIXED ADDRESS", "city": "NO CITY", "zip_code": "123456"}
)
print(customer)

customer_id=100 name='Vishal' address=Address(street='NO FIXED ADDRESS', city='NO CITY', zip_code=123456)


## Pydantic Fields: Customization and Constraints

The Field function in Pydantic enhances model fields beyond basic 'type hints' by allowing you to specify validation rules, default values, aliases, and more.

In [12]:
from pydantic import BaseModel, Field

class Item(BaseModel):
    name:str= Field(min_length= 2, max_length= 50)
    price:float= Field(gt= 100, le= 1000)           # Greater than 0 and less than or equal to 1000
    quantity:int= Field(ge= 0)                      # Greater than or equal to '0'

# Valid instance
item = Item(name= "Book", price= 101, quantity= 10)
print(item)

name='Book' price=101.0 quantity=10


In [13]:
Invalid_Item = Item(name= "Book", price= 100, quantity= -1)
print(Invalid_Item)

ValidationError: 2 validation errors for Item
price
  Input should be greater than 100 [type=greater_than, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than
quantity
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than_equal

In [14]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., description= "Unique username for the user")
    age: int = Field(default= 18, description= "User age defaults to 18")
    email: str = Field(default_factory= lambda: "user@example.com", description= "Default email address")


In [15]:
user1 = User(username="Vishal")
print(user1)

username='Vishal' age=18 email='user@example.com'


In [16]:
user2 = User(username="Goyal", age=25, email="INDIA@domain.com")
print(user2)

username='Goyal' age=25 email='INDIA@domain.com'


In [17]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User age defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
